# មេរៀន 18: ការការពារ <i>AI Agents</i> ជាមួយវិក្កយបត្រការពារ​ឌីជីថល

## សៀវភៅកំណត់ត្រាដែលអនុវត្តបាន

សៀវភៅកំណត់ត្រានេះដឹកនាំតាមវគ្គបួនក្នុងការ​បង្រៀន៖

1. **ចុះហត្ថលេខាលើវិក្កយបត្រដំបូងរបស់អ្នក** សម្រាប់ការហៅឧបករណ៍របស់ភ្នាក់ងារនិងបញ្ជាក់វា។
2. **បំភ្លេចលើវិក្កយបត្រ** ហើយមើលការបញ្ជាក់ធ្លាក់បរាជ័យ។
3. **សាងសង់ខ្សែវិក្កយបត្រ​បីសន្លឹក** ហើយបញ្ជាក់ភាពសុពលភាពនៃខ្សែទង្វើ។
4. **ប្រមួលការហៅឧបករណ៍ Microsoft Agent Framework** ដើម្បីឲ្យសកម្មភាពគ្រប់យ៉ាងបញ្ចេញវិក្កយបត្រឲ្យបាន។

រូបមន្តវិជ្ជាជីវៈសុវត្ថិភាពទាំងអស់ត្រូវបាននាំចូលពីបណ្ណាល័យដែលថែទាំល្អ (`pynacl` សម្រាប់ Ed25519, `jcs` សម្រាប់ RFC 8785 canonical JSON, `hashlib` ពីបណ្ណាល័យស្តង់ដាររបស់ Python សម្រាប់ SHA-256)។ តុល្យភាពវិក្កយបត្រត្រូវបានសរសេរជា Python សាមញ្ញដែលអ្នកអាចអាននិងកែប្រែបាន។

ដំណើរការទំព័រនីមួយៗតាមលំដាប់។ ប្រភាគមួយៗមានបរិមាណខ្លីហើយអាចបន្ទាន់បានដោយខ្លួនឯង។


## ការដំឡើង

តំឡើងការពឹងផ្អែកពីរទាំងពីរ។ ទាំងពីរនេះមានអាជ្ញាប័ណ្ណបង្ហួល (Apache-2.0 / MIT)។  


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## ឧបករណ៍ជួយ

អ្នកជួយទាំងពីរនេះដំណើរការការអ៊ិនគូដ base64url (ដោយគ្មានការបញ្ចូល) និងការបង្កើតអត្ថបទ hashed SHA-256 របស់វត្ថុផ្សេងៗមួយណា។ ពួកគេរក្សាទុកផ្នែកដែលនៅសល់នៃសៀវភៅកំណត់បណ្ដាញផ្តោតលើលោហការនៃបរិញ្ញាបត្រផ្ទាល់។ 


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## ដំណាក់កាលទី 1: ទទួលបានចុះហត្ថលេខាលើបង្កាន់ដៃដំបូងរបស់អ្នក

សូមគំនិតថាអ្នកតំណាងរបស់យើងសម្រាប់ **Contoso Travel** ទើបតែស្វែងរកការហោះហើរពី Sydney ទៅ Los Angeles សម្រាប់អតិថិជនម្នាក់។ យើងចង់កត់ត្រាការហៅឧបករណ៍នេះជាបង្កាន់ដៃដែលបានចុះហត្ថលេខា ដូច្នេះអ្នកត្រួតពិនិត្យនៅពេលក្រោយអាចបញ្ជាក់វាបានដោយគ្មានការជឿទុកចិត្តលើយើង។

### ឈុតជំហាន 1.1៖ បង្កើតកន្ធ្រsignature key

ក្នុងការផលិត ពាក្យសម្ងាត់ការចុះហត្ថលេខារបស់ភ្នាក់ងារនឹងស្ថិតនៅក្នុងម៉ូឌុលសុវត្ថិភាពរឹង (HSM), Azure Key Vault, ឬហាងដែលបានការពារដូចគ្នា។ សម្រាប់មេរៀននេះ យើងបង្កើតកូនសោថ្មីក្នុងអង្គចងចាំ។


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### ជំហានទី 1.2: បង្កើតទិន្នន័យ receipt

ទិន្នន័យនេះមានអ្វីៗគ្រប់យ៉ាងដែលយើងចង់ឲ្យ receipt ផ្ទៀងផ្ទាត់៖ អ្នកដែលបានអនុវត្ត កម្មវិធីដែលបានប្រើ ជាមួយប៉ារ៉ាម៉ែត្រអ្វី អ្វីដែលបានត្រឡប់មកក្រោយ ជា​ផ្នែកនៃ​គោលការណ៍ណា ហើយពេលណា។ យើងធ្វើ hash លើប៉ារ៉ាម៉ែត្រ និងលទ្ធផលជំនួសការរាប់បញ្ចូលពួកវាផ្ទាល់ ដើម្បីឲ្យ receipt មិនបញ្ចេញមាតិកាដែលមានភាពឯកជនបាន។


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### ជំហាន 1.3៖ ហត្ថលេខា និងប្រមូលផ្តុំវិក្កយបត្រ

បីជំហាន:

1. ធ្វើការច្បាស់លាស់លើ payload ដោយប្រើ JCS ដើម្បីឲ្យការអនុវត្តពីរដែលបញ្ចេញវិក្កយបត្រដូចគ្នាយ៉ាងហោចណាស់បង្កើតប៉ុងខ្សែបៃដូចគ្នា។
2. រុញកូដ byte ដោយប្រើ SHA-256។
3. ហត្ថលេខាលើកូដ hash ដោយកូនសោឯកជន Ed25519។

ហត្ថលេខានោះបន្ទាប់មកត្រូវបានភ្ជាប់ទៅនឹង payload ដើមសម្រាប់បង្កើតវិក្កយបត្រចុងក្រោយ។


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### ជំហាន 1.4៖ ពិនិត្យបង្កាន់ដៃ

ការពិនិត្យបង្កាន់ដៃគឺជាការវិលត្រឡប់ដំណើរការ។ យើងដកហត្ថលេខា ធ្វើការគណនាថ្មី hash canonical ហើយពិនិត្យហត្ថលេខាអាច័ត្រជាមួយកូនសោសាធារណៈក្នុងបង្កាន់ដៃ។

អ្នកត្រួតពិនិត្យដែលបំពេញការពិនិត្យនេះ មិនចាំបាច់មានអ្វីពីយើងតែបង្កាន់ដៃនោះទេ។ មិនចាំបាច់ហៅសេវា មិនចាំបាច់សួរទីតាំងកូនសោ មិនចាំបាច់មានការជឿទុកចិត្ត។ 


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

អ្នកគួរតែឃើញ `Receipt is valid: True` ។ ឯកអគ្គរដ្ឋទូតបានផលិតកំណត់ត្រាការត្រួតពិនិត្យដែលបានចុះហត្ថលេខាតាមអាគ្រីប្តូក្រាផ្សងព្រេងដំបូងរបស់ខ្លួន។


## ផ្នែក 2: កែប្រែ​បង្ករ​បា្រក់

ចំណុច​សំខាន់​នៃ​បង្ករ​បា្រក់​គឺ​ថា​វា​មាន​លក្ខណៈ​បង្ហាញ​ការ​កែប្រែ។ យើង​នឹងបញ្ជាក់វា។

យើង​នឹងកែប្រែ​តែ​តួអក្សរ​មួយ​នៅលើ​បង្ករ​បា្រក់ ហើយមើល​ការ​ផ្ទៀងផ្ទាត់​ដែល​បរាជ័យ។


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### តើមានអ្វីកើតឡើងថ្មីៗនេះ?

ពេលដែលយើងបានផ្លាស់ប្តូរ `policy_id` បៃ canonical បានផ្លាស់ប្តូរទៅ។ តម្លៃ SHA-256 នៃបៃទាំងនោះផ្លាស់ប្តូរទៅដែរ។ ហត្ថលេខា (ដែលបានធ្វើលើ hash ដើម) មិនត្រូវគ្នានឹង hash ថ្មីទេ។ ការ​ពិនិត្យ​សម្ងាត់​បានត្រលប់ត្រឹមត្រូវជា `False`។

មិនមានវិធីណាមួយក្នុងការប្រែប្រួលអំពីប្រអប់ណាមួយនៃវិកយបត្រនិងនៅតែអាចផ្ទៀងផ្ទាត់បានទេ លើសតែលោកបោកប្រាស់មាន key ផ្ទាល់ខ្លួន។ បើ key ផ្ទាល់ខ្លួនមាននៅក្នុង key vault និង key សាធារណៈត្រូវបានផ្សាយ ផលប៉ះពាល់នៃការប្រែប្រួលគឺមិនអាចលាក់បានឡើយ។

សាកល្បងដូចខ្លួនឯង៖ ប្រែប្រួល `tool_name` ឬ `agent_id` ឬ `timestamp` នៅក្នុងក្រឡា​ខាងលើ ហើយរត់ម្តងទៀត។ ការផ្លាស់ប្តូរទាំងអស់បង្កើតវិកយបត្រមិនត្រឹមត្រូវ។


## ផ្នែក 3: ចងខលខយបន្ធូរជាមួយគ្នា

បង្កាន់ដៃតែមួយគ្រប់គ្រងសកម្មភាពទាំងអស់មួយ។ភាគច្រើនភ្នាក់ងារធ្វើសកម្មភាពជាច្រើន។ ដើម្បីធ្វើឱ្យលំដាប់ទាំងមូលមានសក្ដានុពលបំភ្លឺការបញ្ជក់ យើងភ្ជាប់បង្កាន់ដៃនីមួយៗជាមួយនឹងបង្កាន់ដៃមុនដោយរួមបញ្ចូលអ៊ុម ហាស (hash) របស់បង្កាន់ដៃមុន ក្នុងទិន្នន័យ payload របស់បង្កាន់ដៃថ្មី។

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

ប្រសិនបើមាននរណាម្នាក់ដក ឬប្តូរឃុំបង្កាន់ដៃ ទង្វើបង្វិលនឹងខូចនៅចំណុចដែលបានបញ្ជាក់។ ការផ្ទៀងផ្ទាត់បង្កាន់ដៃណាមួយក្រោយមកនឹងបរាជ័យ ព្រោះ `previous_receipt_hash` មិនត្រូវនឹង អ៊ុម ហាសពិតប្រាកដរបស់បង្កាន់ដៃមុនទេ។


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

ឥឡូវនេះបំបែកខ្សែខ្នងដោយកម្មវិធីកែប្រែចំលែកបង្កាន់ដៃកណ្តាល ហើយធ្វើការត្រួតពិនិត្យឡើងវិញ។ បង្កាន់ដៃដែលត្រូវបានកែប្រែបរាជ័យក្នុងការត្រួតពិនិត្យហត្ថលេខា រួមទាំងបង្កាន់ដៃបន្ទាប់បរាជ័យក្នុងការត្រួតពិនិត្យខ្សែខ្នង (ព្រោះ `previous_receipt_hash` មិនត្រូវនឹងហាសបង្កាន់ដៃកណ្តាលដែលបានកែប្រែទៀតទេ)។


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

បង្កាន់ដៃ 0 នៅតែផ្ទៀងផ្ទាត់បាន (វាមិនត្រូវបានកែប្រែទេ ហើយគ្មានបណ្ដោយមុនណាមួយដែលអាចពឹងផ្អែកលើវាទេ)។ បង្កាន់ដៃ 1 បរាជ័យការត្រួតពិនិត្យហត្ថលេខារបស់វាដោយសារយើងបានផ្លាស់ប្តូរ `tool_args_hash`។ បង្កាន់ដៃ 2 បរាជ័យការត្រួតពិនិត្យខ្សែចងខ្សែដោយសារតែ `previous_receipt_hash` របស់វាត្រូវបានគណនាពីបង្កាន់ដៃ 1 ដើម (ដែលឥឡូវត្រូវបានកែប្រែ)។

បើទោះបីជាអ្នកប្រហារបានហត្ថលេខាថ្មីលើបង្កាន់ដៃ 1 ដែលបានកែប្រែ (ដែលពួកគេមិនអាចធ្វើបានដោយគ្មានកូនសោឯកជន) ការចំណាត់ថ្នាក់ទំនងខ្សែចងខ្សែក្នុងបង្កាន់ដៃ 2 នៅតែមើលឃើញការបំប្លែង។ ដើម្បីលាក់ការផ្លាស់ប្តូរ អ្នកប្រហារត្រូវតែហត្ថលេខាថ្មីលើបង្កាន់ដៃគ្រប់អំពីចំណុចកែប្រែ តាមដែលទាមទារឲ្យមានការកាន់កាប់កូនសោឯកជន។


## ផ្នែក ៤៖ ការព្រមទាំងឧបករណ៍ភ្នាក់ងារជាមួយការចុះហត្ថលេខាលើប័ណ្ណទទួល

ក្នុងការដាក់ឲ្យដំណើរការពិតប្រាកដ អ្នកមិនចង់ឲ្យអ្នកនិពន្ធភ្នាក់ងារនាល់ម្នាក់ចាំត្រូវហៅ `make_receipt` ឡើយ។ អ្នកចង់ឲ្យការចុះហត្ថលេខាលើប័ណ្ណទទួលជាប្រពៃណីស្វ័យប្រវត្តិក្នុងការហៅឧបករណ៍នីមួយៗ។

នេះជាគំរូដែលសាមញ្ញបំផុត៖ ថ្នាក់លួចបញ្កួចមួយដែលយកមុខងារឧបករណ៍អាចហៅបានមួយណាមួយ ហើយត្រឡប់មកជាប្រភេទដែលបញ្ចេញប័ណ្ណទទួល។ វាអាចអនុវត្តទៅលើស៊ុមភ្នាក់ងារណាមួយបាន ដូចជា Microsoft Agent Framework (`agent_framework.foundry`)។

ប្រសិនបើអ្នកមិនទាន់មានគម្រោង Microsoft Foundry ត្រូវបង្កើតឡើងទេ គំរូថ្នាក់មូលនៅក្នុងកន្លែងក្រោមនេះនៅតែបង្ហាញពីគំរូនោះបាន។


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### រួមបញ្ចូលជាមួយស៊ុមហ្វ្រេម Microsoft Agent

ការវេចខ្ចប់ `ReceiptedTool` ខាងលើគឺមិនអាចអាស្រ័យលើស៊ុមហ្វ្រេមណាមួយទេ។ ដើម្បីប្រើវាក្នុងភ្នាក់ងារដែលបានសាងសង់ជាមួយស៊ុមហ្វ្រេម Microsoft Agent, ចុះបញ្ជីមុខងារដែលបានវេចខ្ចប់ជា ឧបករណ៍មួយ។ គំរូមួយ (អ្នកអាចប្តូរជំនួស mock ជាការចុះបញ្ជីឧបករណ៍ Microsoft Foundry ពិតប្រាកដ):

```python
# កូដក្លែងបង្ហាញពីរូបរាងជំនុំរួម។
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="អ្នកជាអ្នកដឹកនាំការធ្វើដំណើររបស់ Contoso ...",
#     tools=[receipted_lookup],   # ឧបករណ៍ដែលបានដាក់ច្រវ៉ាក់ មិនមែនមុខងារដើមទេ
# )
# response = agent.run("ស្វែងរកបើកជើងហោះហើរពីស៊ីឌនីទៅឡូសអេឡេណហ្ស៍នៅខែមិថុនា។")
#
# # បន្ទាប់ពីដំណើរការ បានហត្ថលេខាទៅលើការហៅឧបករណ៍រាល់ផ្សេងៗដែលភ្នាក់ងារបានធ្វើ៖
# audit_chain = receipted_lookup.receipts
```

ស៊ុមហ្វ្រេមភ្នាក់ងារមិនត្រូវការប្រាប់អ្វីគ្រប់យ៉ាងអំពីបង្កាន់ដៃទេ។ ការចុះហត្ថលេខា Receipt ត្រូវបានវេចខ្ចប់ជុំវិញឧបករណ៍ មិនមែនភ្ជាប់ជាមួយស៊ុមហ្វ្រេមនោះឡើយ។ នេះគឺជាវិធីដែលអ្នកបញ្ចូលប្រភពដើមទៅកូដភ្នាក់ងារដែលមានរួចរហ័សដោយមិនចាំបាច់សរសេរឡើងវិញភ្នាក់ងារ។ 


## សង្ខេប និងការប្រឈមមុខបន្ថែម

អ្នកបាន:

- បង្កើតគូកូនសោ Ed25519 ។
- សង់ និងហត្ថលេខាលើបង្កាន់ដៃសម្រាប់ការហៅឧបករណ៍ភ្នាក់ងារ។
- ពិនិត្យបញ្ជាក់បង្កាន់ដៃដោយអុបទិកតែប្រើកូនសោសាធារណៈ។
- បន្លំបង្កាន់ដៃមួយហើយមើលឃើញការត្រួតពិនិត្យបរាជ័យ។
- សង់ខ្សែសង្វាក់បង្កាន់ដៃចំនួនបី។
- បន្លំផ្នែកកណ្តាលនៃខ្សែសង្វាក់ ហើយមើលឃើញការបរាជ័យទាំងស្គ្រីបហត្ថលេខានិងខ្សែសង្វាក់។
- અવૃત્તិបគ្គងនីយមក្នុងរូបមន្តឧបករណ៍ជាមួយការហត្ថលេខាដោយស្វ័យប្រវត្តិបង្កាន់ដៃ។

**ការប្រឈមបន្ថែម។** ផ្លាស់ប្តូររចនាសម្ព័ន្ធបង្កាន់ដៃជាមួយវាល `request_id` (UUID សម្រាប់ការតាមដានចែកចាយ)។ ប្ដូរ `make_receipt` ដើម្បីរួមបញ្ចូលវា និងបញ្ជាក់ថាបង្កាន់ដៃនៅតែអាចត្រួតពិនិត្យបានពីដើមដល់ចុង។ បន្ទាប់មកកែប្រែវាលនេះបន្ទាប់ពីហត្ថលេខា ហើយបញ្ជាក់ថាការត្រួតពិនិត្យបរាជ័យ។ នេះបង្ខំឲ្យអ្នកយល់ដឹងប្រែប្រួលថាយបាណាតែមួយជាទម្រង់ឯកសារផ្លូវការមានឥទ្ធិពលយ៉ាងដូចម្តេចលើហត្ថលេខា។

**ព្រំកំណត់សំខាន់។** បង្កាន់ដៃបញ្ជាក់បីរឿងតែបីរឿង: ការផ្ដល់អត្តសញ្ញាណ (កូនសោនេះបានហត្ថលេខាលើខ្ទង់នេះ), សុទិដ្ឋិនិយម (ខ្ទង់មិនបានផ្លាស់ប្តូរពីពេលហត្ថលេខា), និងលំដាប់ (បង្កាន់ដៃនេះបានមកបន្ទាប់បង្កាន់ដៃនោះ)។ វាមិនបញ្ជាក់ថាសកម្មភាពភ្នាក់ងារមានភាពត្រឹមត្រូវទេ, ឬថាភស្តុតាងដែលមាននៅក្នុង `policy_id` ត្រូវបានវាយតម្លៃពិតប្រាកដ, ឬថាភ្នាក់ងារបានអនុវត្តតាមច្បាប់ទាំងអស់ទេ។ បង្កាន់ដៃគឺជាគ្រឹះ។ របបគ្រប់គ្រងគឺជាប្រព័ន្ធដែលអ្នកសាងសង់លើវា។

អាន README មេរៀនម្ដងទៀតជាមួយការយកចិត្តទុកដាក់លើព្រំកំណត់នោះ។ កំហុសដែលក្រុមភាគច្រើនធ្វើចំពោះបង្កាន់ដៃគឺគិតថា "យើងមានបង្កាន់ដៃ" មានន័យថា "យើងត្រូវបានគ្រប់គ្រង"។ វាមិនត្រឹមត្រូវទេ។ បង្កាន់ដៃធ្វើឲ្យអាកប្បកិរិយារបស់ភ្នាក់ងារអាចត្រួតពិនិត្យបាន។ វាមិនធ្វើឲ្យវាត្រឹមត្រូវឡើយ។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
